<a href="https://colab.research.google.com/github/eyals208/ML_showcase/blob/main/Food_Vision_Project_Big.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Milestone project: food vision  #

To use mixed precision trainning we need the Nvidia T4 GPU

In [2]:
!nvidia-smi -L
!pip install importlib_resources
!pip install --upgrade --force-reinstall tensorflow
!pip install --upgrade scikit-image

GPU 0: Tesla T4 (UUID: GPU-1dc4252d-d0f6-8536-fd74-e239c5c27928)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.9/572.9 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.2/410.2 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.9/71.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 340.4/340.4 kB 36.5 MB/s

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 91.4 MB/s eta 0:00:00
  Attempting uninstall: scikit-image
    Found existing installation: scikit-image 0.25.2
    Uninstalling scikit-image-0.25.2:
      Successfully uninstalled scikit-image-0.25.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cucim-cu12 26.2.1 requires scikit-image<0.26.0,>=0.19.0, but you have scikit-image 0.26.0 which is incompatible.


In [1]:
# Get some helper funtions

!wget https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/extras/helper_functions.py

--2026-08-30 11:34:13--  https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/main/extras/helper_functions.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10246 (10K) [text/plain]
Saving to: ‘helper_functions.py’

helper_functions.py 100%[===================>]  10.01K  --.-KB/s    in 0s      

2026-08-30 11:34:13 (113 MB/s) - ‘helper_functions.py’ saved [10246/10246]



In [2]:
from helper_functions import create_tensorboard_callback, plot_loss_curves, compare_historys

## Get datasets from tensorflow datasets ##

In [3]:
import tensorflow_datasets as tfds


In [ ]:
(train_data, test_data), ds_info =  tfds.load(name="food101",
                                              split=["train", "validation"],
                                              shuffle_files=True,
                                              as_supervised=True,
                                              with_info=True)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

## Visualizin the data ##

In [ ]:
class_names = ds_info.features["label"].names
class_names[:8]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# take a sample of the train data
train_one_sample = train_data.take(1)
for image, label in train_one_sample:
  print(f'''Image shape: {image.shape},
    Image datatype: {image.dtype},
    traget class: {label},
    class name: {class_names[label.numpy()]}''' )

In [ ]:
import tensorflow as tf
tf.reduce_min(image), tf.reduce_max(image)

#plot an image tensor
import matplotlib.pyplot as plt
plt.imshow(image)
plt.title(class_names[label.numpy()])
plt.axis(False)

## Pre-processing the images ##

* Change dtype from unit8 to `float32`
* In batches make sure tensors are same sized
* Scale/Normalize data to between 0 and 1 instead of 0-255. (not needed when using efficientNetB0 because it includes sacling layer)

In [ ]:
def preprocess_img(image, label, image_shape=244):
  '''
  Converts images datatype to float32 and
  reshpaes image to (image_shape, image_shape, 3)

  Returns
  (float32_image, label) tuple
  '''
  image = tf.image.resize(image, [image_shape, image_shape])
  return tf.cast(image, tf.float32), label


In [ ]:
preprocessed_image = preprocess_img(image, label)[0]
preprocessed_image[0].shape, preprocessed_image[0].dtype

## Batch and prepare datasets ##

See https://www.tensorflow.org/guide/data_performance#best_practice_summary

In [ ]:
#map the preprocessing function to trainning data
train_data = train_data.map(map_func=preprocess_img, num_parallel_calls=tf.data.AUTOTUNE)
#shuffle train data
train_data = train_data.shuffle(buffer_size=1000).batch(batch_size=32).prefetch(buffer_size=tf.data.AUTOTUNE)

#map preprocessing function to test data
test_data = test_data.map(preprocess_img, num_parallel_calls=tf.data.AUTOTUNE).batch(32).prefetch(tf.data.AUTOTUNE)

## Creating callbacks ##

1. Tensorboard callback
2. ModelCheckPoint callback

In [ ]:
# get tensorboard callback from helper_functions
from helper_functions import create_tensorboard_callback
import tensorflow as tf
# Create ModelCheckPoint callback
checkpoitn_path = "model_checkpoints/cp.weights.h5"
model_checkpoint = tf.keras.callbacks.ModelCheckpoint(checkpoitn_path,
                                                      monitor="val_acc",
                                                      mode="max",
                                                      save_best_only=True,
                                                      save_weights_only=True,
                                                      verbose=0)

## Setting up Mixed precision ##

See documentation: https://www.tensorflow.org/guide/mixed_precision

In [ ]:
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")


In [ ]:
mixed_precision.global_policy()

## Create a feature exctraction model

In [ ]:
from os import name
from tensorflow.keras import layers

# create base model
input_shape = (244,244,3)

base_model = tf.keras.applications.EfficientNetB0(include_top=False)
base_model.trainable = False

#create functional model
inputs = layers.Input(shape=input_shape, name="input_layer")
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
#x = layers.Dense(len(class_names))(x)
x = layers.Dense(101)(x)
outputs = layers.Activation("softmax", dtype=tf.float32, name="softmax_float32")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(loss=tf.keras.losses.sparse_categorical_crossentropy,
              optimizer=tf.keras.optimizers.Adam(),
              metrics=["accuracy"])

In [ ]:
# check dtype_policy attributes of layers in out models
for layer in model.layers:
  print(layer.name, layer.trainable, layer.dtype, layer.dtype_policy)

In [ ]:
for layer in base_model.layers[-20:]:
  print(layer.name, layer.trainable, layer.dtype, layer.dtype_policy)

## Fit the model ##

First train the output layer
Then unfreeze some layers of the base model and fine-tune

In [ ]:
history_101_food_classes_feature_extraction = model.fit(train_data,
                                                        epochs=3,
                                                        steps_per_epoch=len(train_data),
                                                        validation_data=test_data,
                                                        validation_steps=int(0.15 * len(test_data)),
                                                        callbacks=[create_tensorboard_callback("training_logs","efficiecntnetb0_101_classes_all_data"),
                                                                   model_checkpoint])

In [ ]:
results_feature_extraction_model = model.evaluate(test_data)
results_feature_extraction_model

## Save the model ##

In [ ]:
%cd /content/ML_showcase/

In [ ]:
#model.save_weights('/content/drive/food_vision_feature_extract/w.weights.h5')
model.save('food_vision_feature_extract_model.keras')


## Converting model to js for embedding in the html page

In [ ]:
# Install the converter utility
!pip install tensorflowjs --quiet

# Convert and save the shards directly into the repo's web assets folder
!tensorflowjs_converter --input_format=keras my_best_model.h5 ../tfjs_models/image_model
